In [19]:
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import random
import os
from torch_geometric.nn import EdgeConv, global_mean_pool
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split

In [20]:
dataset_path = r"C:\Users\HP\Downloads\SiteSpec-GNN\LRN_Dataset.pkl"
with open(dataset_path, "rb") as f:
    dataset = pickle.load(f)

print(f"Loaded dataset with {len(dataset)} graphs")
print(dataset[0])

Loaded dataset with 225 graphs
Data(x=[3, 21], edge_index=[2, 10], edge_attr=[10, 2], y=[1])


In [21]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)
torch.cuda.manual_seed_all(seed)

In [22]:
train_data, test_data = train_test_split(
    range(len(dataset)),
    test_size=0.2,
    random_state=seed,
    stratify=[d.y.item() for d in dataset]
)
train_dataset = [dataset[i] for i in train_data]
test_dataset = [dataset[i] for i in test_data]

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

In [23]:
class EdgeNet(nn.Module):
    def __init__(self, node_in_dim, hidden_dim, num_classes):
        super(EdgeNet, self).__init__()
        self.edge_mlp1 = nn.Sequential(
            nn.Linear(2 * node_in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )       
        self.conv1 = EdgeConv(nn=self.edge_mlp1)
        self.lin_out = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index)      
        x = global_mean_pool(x, batch)
        x = self.lin_out(x)
        return x

In [24]:
node_in_dim = dataset[0].x.shape[1]
hidden_dim = 16
num_classes = len(set(d.y.item() for d in dataset))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = EdgeNet(node_in_dim, hidden_dim, num_classes).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [25]:
def train():
    model.train()
    total_loss, correct, total = 0, 0, 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch.x, batch.edge_index, batch.batch)
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pred = out.argmax(dim=1)
        correct += (pred == batch.y).sum().item()
        total += batch.num_graphs
    return total_loss / len(train_loader), correct / total

In [26]:
def test(loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch)
            loss = criterion(out, batch.y)
            total_loss += loss.item()
            pred = out.argmax(dim=1)
            correct += (pred == batch.y).sum().item()
            total += batch.num_graphs
    return total_loss / len(loader), correct / total

In [27]:
train_losses, test_losses = [], []
train_accs, test_accs = [], []

for epoch in range(1, 301):
    train_loss, train_acc = train()
    test_loss, test_acc = test(test_loader)

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    
best_test_acc = max(test_accs)
best_test_epoch = test_accs.index(best_test_acc) + 1

train_acc_at_best_test = train_accs[best_test_epoch - 1]

print(f"\nTrain Accuracy: {train_acc_at_best_test:.4f}")
print(f"Test Accuracy: {best_test_acc:.4f} ")




Train Accuracy: 0.9611
Test Accuracy: 0.9556 
